# ETHICS Religion — gambar Bab VI pada lebar 14 cm

## SPEC dan DESIGN PLAN

Lihat `docs/ethics_religion_bab6_14cm_20261008.md`. Dua kelompok: Ateis–Kristen dan Islam–Yahudi. Setiap eksekusi membuat run baru; tidak menjalankan notebook lama, menimpa PNG aktif, mengubah data atau mengulang statistik. Akurasi tetap per subset. Efek memakai conditional accuracy persona−N keluarga Holm 16, bukan hasil heatmap 17 atau prasyarat Q dari gender.


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib, json, textwrap
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.text import Text

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/"notebooks/religion/ethics accuracy analysis/tables").is_dir())
ACTIVE = ROOT/"notebooks/religion/ethics accuracy analysis"
TABLES = ACTIVE/"tables"
RUN = ROOT/"result/analyses/accuracy/runs"/("religion_ethics_bab6_14cm_"+datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
RUN.mkdir(parents=True,exist_ok=False)
FIG=RUN/"bab6"; FIG.mkdir()
SUP=RUN/"tables"; SUP.mkdir()
EXTRA=RUN/"pelengkap_committed"; EXTRA.mkdir()
PROOF=RUN/"proof"; PROOF.mkdir()
def digest(p): return hashlib.sha256(p.read_bytes()).hexdigest()
protected=list(TABLES.glob("*"))+list((ACTIVE/"figures").glob("*"))+list(ACTIVE.glob("*.ipynb"))
baseline={str(p.relative_to(ROOT)):digest(p) for p in protected if p.is_file()}
def read(name): return pd.read_csv(TABLES/(name+".csv"))
summary=read("ethics_condition_summary_source_derived")
parity=read("ethics_religion_parity_condition_summary")
all_status=read("ethics_religion_parity_all_subset_summary")
category=read("ethics_religion_parity_category_summary")
reasons=read("ethics_religion_parity_abstain_reasons")
pairs=read("ethics_religion_parity_paired_effects_vs_neutral")
effects=pairs[pairs.outcome.eq("conditional_accuracy")].copy()
old_combined=read("ethics_religion_combined_four_subsets")
models=["qwen_qwen3-next-80b-a3b-thinking","meta-llama_llama-3.1-8b-instruct","mistralai_ministral-8b-2512","deepseek_deepseek-r1-distill-qwen-32b","openai_gpt-oss-20b","meta-llama_llama-3.3-70b-instruct"]
names=["Qwen3 Next 80B-A3B","Llama 3.1 8B","Ministral 8B","DeepSeek R1-Distill 32B","GPT-OSS 20B","Llama 3.3 70B"]
model_names=dict(zip(models,names))
identity_labels={"atheist":"Ateis","christian":"Kristen","islam":"Islam","jewish":"Yahudi"}
groups=[("ateis_kristen",["atheist","christian"]),("islam_yahudi",["islam","jewish"])]
scenarios=["explicit","implicit_1","implicit_2","implicit_3"]
scenario_labels=["E","I1","I2","I3"]
def personas_for(ids): return [i+"_"+s for i in ids for s in scenarios]
def labels_for(ids): return [identity_labels[i]+"\n"+s for i in ids for s in scenario_labels]
plt.rcParams.update({"font.family":"DejaVu Sans","font.size":8,"axes.titlesize":9,"axes.labelsize":8,"xtick.labelsize":7,"ytick.labelsize":7,"legend.fontsize":7,"pdf.fonttype":42,"savefig.facecolor":"white"})
WIDTH=14/2.54
checks={}; captions={}; artifacts=[]; cell_audit=[]; figure_audit=[]
assert len(summary)==408 and summary.total.sum()==154428
assert len(effects)==192 and effects.family_size.eq(16).all()
assert set(effects.subset)=={"commonsense","justice"}
assert effects.persona_neutral.eq("neutral").all()
assert not effects.duplicated(["model","subset","persona"]).any()
assert (effects.significant_holm_0_05 == effects.holm_adjusted_p_value.lt(.05)).all()
np.testing.assert_allclose(effects.risk_difference_pp,effects.rate_a_pct-effects.rate_b_pct,atol=1e-12)
for frame in [summary,all_status,category]:
    assert (frame.total==frame.committed+frame.abstain+frame.missing).all()
    assert (frame.committed==frame.n_answer_wrong+frame.n_answer_acceptable).all()
    np.testing.assert_allclose(frame.commit_rate,frame.committed/frame.total)
    np.testing.assert_allclose(frame.pct_answer_wrong,frame.n_answer_wrong/frame.committed)
    np.testing.assert_allclose(frame.pct_answer_acceptable,frame.n_answer_acceptable/frame.committed)
# Preserve source valid-subset values, with invalid-subset accuracy omitted.
valid=summary[summary.subset.isin(["commonsense","justice"])].copy()
np.testing.assert_allclose(valid.accuracy_committed,valid.n_correct/valid.committed)
keys=["model","persona","subset"]
pd.testing.assert_frame_equal(valid.set_index(keys).sort_index()[["accuracy_committed","commit_rate"]],
    parity[parity.subset.isin(["commonsense","justice"])].set_index(keys).sort_index()[["accuracy_committed","commit_rate"]])
count_cols=["total","committed","abstain","missing","n_answer_wrong","n_answer_acceptable"]
pd.testing.assert_frame_equal(summary.groupby(["model","persona"])[count_cols].sum().sort_index(),
    all_status.set_index(["model","persona"])[count_cols].sort_index(),check_dtype=False)
pool=all_status.copy()
pool["category"]=pool.persona.str.replace(r"_implicit_[123]$","_implicit_avg",regex=True)
pd.testing.assert_frame_equal(pool.groupby(["model","category"])[count_cols].sum().sort_index(),
    category.set_index(["model","category"])[count_cols].sort_index(),check_dtype=False)
# Cross-check unchanged counts against the previous combined figure table.
for column,old in [("total","total"),("committed","committed"),("abstain","abstain"),("missing","missing"),("n_answer_wrong","n_wrong"),("n_answer_acceptable","n_acceptable")]:
    a=category.set_index(["model","category_short"])[column].sort_index()
    b=old_combined.set_index(["model","category_short"])[old].sort_index()
    pd.testing.assert_series_equal(a,b,check_names=False)
reason_totals=reasons.groupby(["model","persona_folder"]).reason_count.sum().rename_axis(["model","persona"])
pd.testing.assert_series_equal(reason_totals.sort_index(),all_status.set_index(["model","persona"]).abstain.sort_index(),check_names=False)
np.testing.assert_allclose(reasons.pct_of_abstain,100*reasons.reason_count/reasons.abstain)
checks.update({"source_rows":int(summary.total.sum()),"status_counts":summary[["committed","abstain","missing"]].sum().astype(int).to_dict(),"unique_accuracy_cells":len(valid),"unique_effect_cells":len(effects),"holm_significant_cells":int(effects.significant_holm_0_05.sum()),"holm_family_size":16,"source_counts_and_pooling_verified":True,"reason_counts_exhaust_abstain":True})
valid.to_csv(SUP/"accuracy_per_subset_with_coverage.csv",index=False)
effects.to_csv(SUP/"conditional_accuracy_persona_vs_N_source.csv",index=False)
all_status.drop(columns=["n_correct","accuracy_committed","effective_success_rate_nonmissing"]).to_csv(SUP/"all_subset_status_denominators.csv",index=False)
category.drop(columns=["n_correct","accuracy_committed","effective_success_rate_nonmissing"]).to_csv(SUP/"category_counts_with_coverage.csv",index=False)

def dec(v,digits=1,signed=False):
    return (f"{v:+.{digits}f}" if signed else f"{v:.{digits}f}").replace(". ",",").replace(".",",").replace("-","−")

def heat(ax,data,xlabels,ylabels,vmin=0,vmax=100,cmap="YlOrBr",annotations=None,fontsize=7.2):
    data=np.asarray(data,dtype=float)
    im=ax.pcolormesh(np.arange(data.shape[1]+1)-.5,np.arange(data.shape[0]+1)-.5,np.ma.masked_invalid(data),cmap=cmap,vmin=vmin,vmax=vmax,shading="flat",rasterized=False)
    ax.set_ylim(data.shape[0]-.5,-.5); ax.set_xlim(-.5,data.shape[1]-.5)
    ax.set_xticks(range(data.shape[1]),xlabels); ax.set_yticks(range(data.shape[0]),ylabels)
    ax.tick_params(length=0)
    ax.set_xticks(np.arange(-.5,data.shape[1],1),minor=True); ax.set_yticks(np.arange(-.5,data.shape[0],1),minor=True)
    ax.grid(which="minor",color="white",linewidth=.5); ax.tick_params(which="minor",bottom=False,left=False)
    for s in ax.spines.values(): s.set_visible(False)
    for (i,j),v in np.ndenumerate(data):
        label=annotations[i,j] if annotations is not None else dec(v) if np.isfinite(v) else "NA"
        rgba=im.cmap(im.norm(v)) if np.isfinite(v) else (1,1,1,1)
        lum=.2126*rgba[0]+.7152*rgba[1]+.0722*rgba[2]
        t=ax.text(j,i,label,ha="center",va="center",fontsize=fontsize,color="white" if lum<.5 else "#202020")
        t._cell_annotation=True
    return im

def colorbar(fig,im,label,y=.105,ticks=None):
    cax=fig.add_axes([.34,y,.60,.022])
    cb=fig.colorbar(im,cax=cax,orientation="horizontal",label=label,ticks=ticks)
    if cb.solids is not None: cb.solids.set_rasterized(False)
    return cb

def save(fig,stem,caption,directory=FIG):
    assert np.isclose(fig.get_figwidth()*2.54,14)
    fig.canvas.draw(); renderer=fig.canvas.get_renderer()
    for ax in fig.axes:
        for t in ax.texts:
            if getattr(t,"_cell_annotation",False):
                b=t.get_window_extent(renderer); x,y=t.get_position()
                p0=ax.transData.transform((x-.5,y-.5));p1=ax.transData.transform((x+.5,y+.5))
                fits=b.width<abs(p1[0]-p0[0]) and b.height<abs(p1[1]-p0[1])
                cell_audit.append({"figure":stem,"text":t.get_text(),"fits":bool(fits)})
                assert fits,(stem,t.get_text())
        for axis in [ax.xaxis,ax.yaxis]:
            boxes=[t.get_window_extent(renderer) for t in axis.get_ticklabels() if t.get_visible() and t.get_text()]
            assert all(not a.overlaps(b) for a,b in zip(boxes,boxes[1:])),(stem,"tick overlap")
    for t in fig.findobj(Text):
        if t.get_visible() and t.get_text():
            b=t.get_window_extent(renderer)
            assert b.x0>=-1 and b.x1<=fig.bbox.width+1 and b.y0>=-1 and b.y1<=fig.bbox.height+1,(stem,"text outside figure",t.get_text())
    for ext in ["png","pdf"]:
        path=directory/(stem+"."+ext); assert not path.exists()
        fig.savefig(path,dpi=300); artifacts.append(path)
    captions[stem]=caption
    figure_audit.append({"stem":stem,"folder":directory.name,"width_cm":float(fig.get_figwidth()*2.54),"height_cm":float(fig.get_figheight()*2.54)})
    plt.close(fig); print("Selesai:",stem,flush=True)

print("Run:",RUN,flush=True)


## IMPLEMENTATION — akurasi dan efek persona−N

N diulang identik pada dua heatmap akurasi. Tidak ada kolom N−N pada heatmap efek. Pemecahan tampilan tidak memecah ulang keluarga koreksi Holm.


In [ ]:
seen_accuracy=[]; seen_effects=[]
effect_limit=max(1,float(effects.risk_difference_pp.abs().max()))
for tag,ids in groups:
    selected=personas_for(ids)+["neutral"]
    labels=labels_for(ids)+["N"]
    group_title="–".join(identity_labels[i] for i in ids)
    fig,axes=plt.subplots(2,1,figsize=(WIDTH,5.8))
    fig.subplots_adjust(left=.31,right=.98,bottom=.20,top=.88,hspace=.48)
    for ax,subset in zip(axes,["commonsense","justice"]):
        d=valid[valid.subset.eq(subset)].set_index(["model","persona"])
        values=d.accuracy_committed.unstack().reindex(index=models,columns=selected)*100
        cov=d.commit_rate.unstack().reindex(index=models,columns=selected)*100
        ann=np.array([[dec(v)+"\nc "+dec(c) for v,c in zip(vrow,crow)] for vrow,crow in zip(values.to_numpy(),cov.to_numpy())])
        im=heat(ax,values,labels,names,cmap="RdYlGn",annotations=ann,fontsize=7)
        ax.set_title(subset.title(),pad=5)
        ax.tick_params(axis="y",labelsize=6.8);ax.tick_params(axis="x",labelsize=7,pad=4)
        seen_accuracy.extend((subset,model,persona) for model in models for persona in selected)
    fig.suptitle("ETHICS Religion: akurasi committed\n"+group_title+" dan N",fontsize=10,y=.985)
    colorbar(fig,im,"Akurasi (% dari committed)",ticks=[0,25,50,75,100])
    fig.text(.5,.017,"Baris bawah: c = commit rate (% seluruh respons).",ha="center",fontsize=7)
    save(fig,"06_accuracy_"+tag,
         "Akurasi committed, "+group_title+", pada Commonsense dan Justice secara terpisah. Warna dan angka baris atas menunjukkan jumlah benar / committed ×100. Baris bawah c menunjukkan committed / total ×100. Abstain dan missing tidak masuk penyebut akurasi, tetapi masuk total coverage. N adalah netral model yang sama, diulang identik pada kedua gambar sebagai pembanding, bukan observasi tambahan. E = eksplisit; I1–I3 adalah variasi isi warmup, bukan tingkat intensitas. Tabel accuracy_per_subset_with_coverage.csv memuat seluruh hitungan dan rate. Tidak ada akurasi gabungan lintas subset.")
    selected=personas_for(ids);labels=labels_for(ids)
    fig,axes=plt.subplots(2,1,figsize=(WIDTH,5.55))
    fig.subplots_adjust(left=.31,right=.98,bottom=.20,top=.88,hspace=.48)
    for ax,subset in zip(axes,["commonsense","justice"]):
        d=effects[effects.subset.eq(subset)].set_index(["model","persona"])
        values=d.risk_difference_pp.unstack().reindex(index=models,columns=selected)
        sig=d.significant_holm_0_05.unstack().reindex(index=models,columns=selected)
        ann=np.array([[dec(v,1,True)+("\n*" if s else "") for v,s in zip(vrow,srow)] for vrow,srow in zip(values.to_numpy(),sig.to_numpy())])
        im=heat(ax,values,labels,names,vmin=-effect_limit,vmax=effect_limit,cmap="RdBu_r",annotations=ann)
        ax.set_title(subset.title(),pad=5)
        ax.tick_params(axis="y",labelsize=6.8);ax.tick_params(axis="x",labelsize=7,pad=4)
        seen_effects.extend((subset,model,persona) for model in models for persona in selected)
    fig.suptitle("ETHICS Religion: efek akurasi persona − N\n"+group_title,fontsize=10,y=.985)
    colorbar(fig,im,"Delta akurasi (poin persentase)",ticks=[-20,-10,0,10,20])
    fig.text(.5,.017,"* p Holm < 0,05; keluarga tetap 16 kontras/model/subset.",ha="center",fontsize=7)
    save(fig,"11_effect_persona_vs_N_"+tag,
         "Perbedaan akurasi persona−N untuk "+group_title+", pada pasangan soal yang kedua responsnya committed. Angka dan warna adalah risk difference dalam poin persentase, bukan Cohen’s h dan bukan selisih dua akurasi marginal pada gambar 06. Positif berarti persona lebih tinggi daripada N. * berarti p Exact McNemar setelah Holm <0,05, dengan keluarga asli 16 kontras per model/subset/outcome, tidak dikoreksi ulang menjadi delapan. Skala simetris identik pada dua gambar dan mempertahankan batas sumber. Tidak ada prasyarat Q atau dagger tambahan pada tabel ini. n pasangan dan jumlah pasangan discordant tersedia di conditional_accuracy_persona_vs_N_source.csv. Grafik ini tidak menggantikan inferensi seluruh pasangan, kontras I−E, atau antaragama dari tabel lain.")
assert len(set(seen_accuracy))==204 and len(seen_accuracy)==216
assert len(set(seen_effects))==len(seen_effects)==192
checks["all_accuracy_and_effect_cells_preserved"]=True
checks["neutral_duplicate_cells"]=12
checks["effect_scale_pp"]=[-effect_limit,effect_limit]


## IMPLEMENTATION — distribusi total dan committed dipisahkan


In [ ]:
status_colors={"Wrong":"#B80F2E","Acceptable":"#0086A8","Abstain":"#E9C46A","Missing":"#EE6C4D"}
plot_distribution=[]
for tag,ids in groups:
    selected=[i+"_"+s for i in ids for s in ["explicit","implicit_avg"]]+["neutral"]
    labels=[identity_labels[i]+"\n"+s for i in ids for s in ["E","I"]]+["N"]
    group_title="–".join(identity_labels[i] for i in ids)
    for committed_only in [False,True]:
        fig,axes=plt.subplots(3,2,figsize=(WIDTH,6.6))
        fig.subplots_adjust(left=.10,right=.98,bottom=.13,top=.81,hspace=.65,wspace=.27)
        columns=[("n_answer_wrong","Wrong"),("n_answer_acceptable","Acceptable")]
        if not committed_only: columns += [("abstain","Abstain"),("missing","Missing")]
        for ax,model in zip(axes.flat,models):
            frame=category[category.model.eq(model)].set_index("category").loc[selected]
            denominator=frame.committed if committed_only else frame.total
            bottom=np.zeros(5)
            for column,label in columns:
                values=100*frame[column]/denominator.replace(0,np.nan)
                ax.bar(range(5),values,bottom=bottom,width=.8,color=status_colors[label],label=label)
                for j,v in enumerate(values):
                    if np.isfinite(v) and v>=8: ax.text(j,bottom[j]+v/2,dec(v),ha="center",va="center",fontsize=6.8,color="white" if label in ["Wrong","Acceptable"] else "#202020")
                    plot_distribution.append({"figure_type":"committed" if committed_only else "total","model":model,"category":selected[j],"response_type":label,"count":int(frame[column].iloc[j]),"denominator":int(denominator.iloc[j]),"percent":v})
                bottom += values.to_numpy()
            np.testing.assert_allclose(bottom,100)
            ax.set_ylim(0,100);ax.set_yticks([0,50,100]);ax.set_xticks(range(5),labels,fontsize=6.8)
            title=model_names[model].replace("DeepSeek R1-Distill 32B","DeepSeek R1-Distill\n32B")
            ax.set_title(title,fontsize=8)
            ax.set_ylabel("% dari committed" if committed_only else "% seluruh respons",fontsize=7)
        fig.legend(*axes[0,0].get_legend_handles_labels(),loc="upper center",bbox_to_anchor=(.5,.895),ncol=len(columns),frameon=False)
        title="distribusi pada committed" if committed_only else "distribusi seluruh respons"
        fig.suptitle("ETHICS Religion: "+title+"\n"+group_title+" — empat subset",fontsize=10,y=.986)
        fig.text(.5,.027,"I = gabungan hitungan I1–I3; N = netral.\nWrong = keputusan tindakan salah, bukan jawaban model salah.",ha="center",fontsize=7)
        stem=("00_committed_distribution_" if committed_only else "00_total_response_distribution_")+tag
        save(fig,stem,
             "Distribusi "+("Wrong/Acceptable dengan penyebut committed" if committed_only else "Wrong, Acceptable, Abstain, dan Missing dengan penyebut seluruh respons")+" pada empat subset ETHICS, untuk "+group_title+". Setiap batang berjumlah 100%. Hitungan I menggabungkan I1–I3, bukan rata-rata sederhana persentase. Wrong berarti keputusan bahwa tindakan salah, bukan kesalahan jawaban terhadap label. Bukan akurasi gabungan empat subset. N diulang identik sebagai pembanding visual; tidak dihitung dua kali dalam total data. "+("Persentase ini berbeda dari distribusi empat status karena denominatornya berbeda." if committed_only else "Abstain dan missing tidak digabung ke Wrong. Grafik dua status pada committed disimpan terpisah di pelengkap_committed/."),directory=EXTRA if committed_only else FIG)
pd.DataFrame(plot_distribution).to_csv(SUP/"distribution_plot_counts_and_denominators.csv",index=False)


## IMPLEMENTATION — alasan abstain

Empat gambar: dua kelompok agama × dua kelompok tiga model. Seluruh skenario dipertahankan. Alasan diterjemahkan melalui kamus label yang sama dengan gender, tanpa mengubah klasifikasi.


In [ ]:
reason_labels={"both_sides":"Kedua sisi / tidak berpihak","analysis_without_verdict":"Analisis tanpa verdict","no_opinion_stop":"Tidak punya opini lalu berhenti","explicit_binary_refusal":"Menolak jawaban biner","task_substitution":"Mengganti tugas","ai_refusal":"Penolakan sebagai AI","insufficient_context":"Konteks dianggap kurang","placeholder_confusion":"Bingung placeholder","returns_question":"Mengulang pertanyaan","capability_mismatch":"Mengaku tidak mampu","detector_disagreement":"Detektor tidak sepakat","conditional_only":"Jawaban hanya bersyarat"}
assert set(reasons.abstain_reason)==set(reason_labels)
# Match the fixed reason order used in the gender publication package.
reason_order=list(reason_labels)
reason_values=[]
for tag,ids in groups:
    selected=personas_for(ids)+["neutral"]
    labels=labels_for(ids)+["N"]
    group_title="–".join(identity_labels[i] for i in ids)
    for part in range(2):
        fig,axes=plt.subplots(3,1,figsize=(WIDTH,7.65))
        fig.subplots_adjust(left=.39,right=.98,bottom=.14,top=.92,hspace=.46)
        for ax,model in zip(axes,models[part*3:part*3+3]):
            frame=reasons[reasons.model.eq(model)]
            counts=frame.pivot(index="abstain_reason",columns="persona_folder",values="reason_count").reindex(index=reason_order,columns=selected).fillna(0)
            denominator=all_status[all_status.model.eq(model)].set_index("persona").loc[selected,"abstain"]
            matrix=100*counts.div(denominator.replace(0,np.nan),axis=1)
            tick_labels=[label+"\nn="+str(int(n)) for label,n in zip(labels,denominator)]
            im=heat(ax,matrix,tick_labels,[reason_labels[r] for r in reason_order],fontsize=6.8)
            ax.tick_params(axis="x",labelsize=6.4,pad=3);ax.tick_params(axis="y",labelsize=6.5,pad=3)
            ax.set_title(model_names[model],fontsize=8,pad=4)
            for reason in reason_order:
                for persona in selected:
                    reason_values.append({"group":tag,"model":model,"persona":persona,"abstain_reason":reason,"reason_label":reason_labels[reason],"reason_count":int(counts.loc[reason,persona]),"abstain":int(denominator[persona]),"percent":matrix.loc[reason,persona]})
        fig.suptitle("ETHICS Religion: alasan abstain\n"+group_title+f" ({part+1}/2)",fontsize=10,y=.988)
        colorbar(fig,im,"% dari abstain",y=.045,ticks=[0,25,50,75,100])
        save(fig,f"04_abstain_{tag}_{part+1}",
             "Komposisi alasan di antara respons abstain untuk "+group_title+f", kelompok model {part+1}/2. Penyebut adalah n abstain per model/skenario (dicantumkan pada setiap kolom), gabungan empat subset. Skala warna 0–100% sama untuk semua gambar/model. Kategori alasan eksklusif pada tabel sumber. n=0, bila ada, ditampilkan NA; tidak dianggap nol persen. N diulang identik pada kelompok agama, bukan respons tambahan. E = eksplisit; I1–I3 tetap terpisah. Terjemahan label tidak mengubah aturan atau hasil klasifikasi.")
reason_plot=pd.DataFrame(reason_values)
reason_plot.to_csv(SUP/"abstain_reason_plot_values.csv",index=False)
unique_reason=reason_plot.drop_duplicates(["model","persona","abstain_reason"])
assert len(unique_reason)==6*17*12
assert len(reason_plot)==6*18*12
lookup=unique_reason.set_index(["model","persona","abstain_reason"])
for row in reasons.itertuples():
    actual=lookup.loc[(row.model,row.persona_folder,row.abstain_reason)]
    assert actual.reason_count==row.reason_count and actual.abstain==row.abstain
    np.testing.assert_allclose(actual.percent,row.pct_of_abstain)
checks["all_reason_percentages_equal_source"]=True
checks["unique_reason_cells"]=len(unique_reason)
checks["reason_cells_with_repeated_neutral"]=len(reason_plot)
checks["abstain_color_scale"]=[0,100]


## CHECKLIST, TESTING, dan INTEGRATION

Seluruh nilai, cakupan, keluarga Holm, denominator dan hash sumber diperiksa. PDF individual vektor, tepat 14 cm. Proof A4 adalah preview ukuran cetak; bukan bukti cetak fisik. Tidak ada pemasangan `.tex`.


In [ ]:
assert len(captions)==12 and len(artifacts)==24
assert all(p.stat().st_size>0 for p in artifacts)
assert all(digest(ROOT/p)==h for p,h in baseline.items())
checks["protected_files_unchanged"]=len(baseline)
checks["all_heatmap_annotation_boxes_fit"]=all(a["fits"] for a in cell_audit)
checks["all_tick_labels_nonoverlapping"]=True
checks["all_text_within_figure"]=True
checks["figure_count"]=len(captions)
checks["exact_width_cm"]=14
pd.DataFrame(cell_audit).to_csv(SUP/"heatmap_cell_layout_audit.csv",index=False)
pd.DataFrame(figure_audit).to_csv(SUP/"figure_layout_audit.csv",index=False)
proof_stems=["04_abstain_ateis_kristen_1","06_accuracy_ateis_kristen","11_effect_persona_vs_N_islam_yahudi"]
with PdfPages(PROOF/"A4_gambar_padat_14cm.pdf") as pdf:
    for stem in proof_stems:
        img=plt.imread(FIG/(stem+".png"));h=WIDTH*img.shape[0]/img.shape[1]
        fig=plt.figure(figsize=(21/2.54,29.7/2.54))
        ax=fig.add_axes([.16,1-(1+h)/fig.get_figheight(),WIDTH/fig.get_figwidth(),h/fig.get_figheight()])
        ax.imshow(img);ax.axis("off")
        fig.text(.16,.956,"Proof A4 — gambar 14 cm; cetak 100%",fontsize=10)
        brief=" ".join(captions[stem].split()[:65])+"…"
        fig.text(.16,1-(1.12+h)/fig.get_figheight(),textwrap.fill(brief,92),va="top",fontsize=9)
        pdf.savefig(fig);plt.close(fig)
(RUN/"CAPTIONS.md").write_text("# Caption ETHICS religion — 14 cm\n\n"+"\n\n".join("## "+s+"\n\n"+c for s,c in captions.items())+"\n",encoding="utf-8")
guide="# ETHICS religion — gambar 14 cm\n\nDua kelompok: Ateis–Kristen dan Islam–Yahudi. Akurasi per subset; efek tetap conditional accuracy persona−N, keluarga Holm 16.\n\n## Bab VI / kandidat Bab VI\n\n"
for row in figure_audit:
    if row["folder"]=="bab6":guide+="- ["+row["stem"]+".png](bab6/"+row["stem"]+".png)\n"
guide+="\n## Distribusi committed terpisah\n\n"
for row in figure_audit:
    if row["folder"]!="bab6":guide+="- ["+row["stem"]+".png](pelengkap_committed/"+row["stem"]+".png)\n"
guide+="\nPDF dengan nama sama tersedia berdampingan dengan PNG. Caption: CAPTIONS.md. Coverage/denominator/n pasangan: tables/. Proof digital ukuran cetak: proof/A4_gambar_padat_14cm.pdf. N diulang sebagai referensi, bukan observasi tambahan.\n\nAngka sumber, hasil uji, dan aset lama tetap. Persentase distribusi total memang berbeda dari distribusi committed karena penyebut berbeda. Tidak menerapkan syarat Q dari analisis lain, tidak menambahkan kontras baru, tidak mengganti heatmap I−E/antaragama yang sudah tersedia, tidak mengubah .tex.\n\nGambar 08 Deontology/Virtue dan grafik lain yang tidak diminta tidak dirender ulang pada paket ini. Validasi ini mencakup presentasi, bukan audit ulang parser/statistik atau cetak fisik.\n"
(RUN/"MULAI_DI_SINI.md").write_text(guide,encoding="utf-8")
(RUN/"checks.json").write_text(json.dumps(checks,indent=2),encoding="utf-8")
(RUN/"manifest.json").write_text(json.dumps({"run":str(RUN),"source_hashes":baseline,"figure_hashes":{str(p.relative_to(RUN)):digest(p) for p in artifacts},"statistical_tests_rerun":False,"parser_changed":False,"thesis_modified":False,"created_at":datetime.now().isoformat()},indent=2),encoding="utf-8")
print(json.dumps(checks,indent=2))
print("SELESAI:",RUN)
